# keplerv — AI 3D reconstruction (Shap-E diffusion model)

Turns up to six vehicle exterior photos into a real 3D mesh using **Shap-E**, OpenAI's image-to-3D diffusion model (via Hugging Face `diffusers`). The notebook arranges the views into one reference board because Shap-E accepts one image condition, then exposes an API over ngrok for keplerv.com. The result is a rough generative reconstruction, not a photogrammetry-accurate scan.

**Before you run:**
1. `Runtime` → `Change runtime type` → set Hardware accelerator to **T4 GPU** → Save.
2. Get a free ngrok authtoken at https://dashboard.ngrok.com/get-started/your-authtoken (free account, no credit card) and paste it when cell 4 asks for it.
3. `Runtime` → `Run all`. First run downloads model weights (a few GB) — expect several minutes.
4. Copy the public URL printed by the last cell and paste it into the website's AI 3D tab.

**This only works while this notebook keeps running.** Free Colab GPU sessions disconnect after a period of idle time or ~12h max.

In [ ]:
# 1. Install dependencies
!pip install -q diffusers transformers accelerate trimesh pyngrok fastapi "uvicorn[standard]" nest-asyncio python-multipart pillow requests

In [ ]:
# 2. Load the Shap-E image-to-3D diffusion pipeline (downloads weights on first run)
import torch
from diffusers import ShapEImg2ImgPipeline

assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > T4 GPU, then rerun."

pipe = ShapEImg2ImgPipeline.from_pretrained("openai/shap-e-img2img", torch_dtype=torch.float16)
pipe = pipe.to("cuda")
print("Shap-E loaded on", torch.cuda.get_device_name(0))

In [ ]:
# 3. Multi-photo reconstruction: image URLs -> contact sheet -> GLB mesh bytes
import io
from urllib.parse import urlparse
import requests
from PIL import Image, ImageOps, ImageDraw
import trimesh
import numpy as np

ALLOWED_IMAGE_HOSTS = {"keplerv.com", "www.keplerv.com"}

def make_reference_board(image_urls: list[str]) -> Image.Image:
    if not image_urls or len(image_urls) > 6:
        raise ValueError("Provide between 1 and 6 exterior photo URLs.")
    board = Image.new("RGB", (1024, 1024), "white")
    tile_size, columns = 304, 3
    grid_height = 2 * tile_size
    top = (1024 - grid_height) // 2
    for i, url in enumerate(image_urls):
        parsed = urlparse(url)
        if parsed.scheme != "https" or parsed.hostname not in ALLOWED_IMAGE_HOSTS:
            raise ValueError("Photo URLs must be HTTPS keplerv.com image URLs.")
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        image = Image.open(io.BytesIO(resp.content)).convert("RGB")
        image.thumbnail((tile_size - 12, tile_size - 12), Image.Resampling.LANCZOS)
        tile = Image.new("RGB", (tile_size, tile_size), "white")
        tile.paste(image, ((tile_size-image.width)//2, (tile_size-image.height)//2))
        x = (i % columns) * tile_size + 56
        y = top + (i // columns) * tile_size
        board.paste(tile, (x, y))
    return board

def reconstruct_to_glb(image_urls: list[str], steps: int = 64, guidance_scale: float = 3.0) -> bytes:
    board = make_reference_board(image_urls)
    result = pipe(image=board, guidance_scale=guidance_scale, num_inference_steps=steps, output_type="mesh")
    mesh_out = result.images[0]
    verts = mesh_out.verts.detach().cpu().numpy() if torch.is_tensor(mesh_out.verts) else np.asarray(mesh_out.verts)
    faces = mesh_out.faces.detach().cpu().numpy() if torch.is_tensor(mesh_out.faces) else np.asarray(mesh_out.faces)
    vertex_colors = None
    if getattr(mesh_out, "vertex_channels", None):
        try:
            rgb = np.stack([mesh_out.vertex_channels[c].detach().cpu().numpy() for c in "RGB"], axis=1)
            vertex_colors = np.clip(rgb * 255, 0, 255).astype(np.uint8)
        except Exception as e:
            print("no vertex colors:", e)
    tri_mesh = trimesh.Trimesh(vertices=verts, faces=faces, vertex_colors=vertex_colors, process=False)
    buf = io.BytesIO()
    tri_mesh.export(buf, file_type="glb")
    return buf.getvalue()

In [ ]:
# 4. ngrok auth — paste your free token from https://dashboard.ngrok.com/get-started/your-authtoken
from getpass import getpass
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = getpass("Paste your ngrok authtoken: ")
conf.get_default().auth_token = NGROK_AUTHTOKEN

In [ ]:
# 5. FastAPI server + public tunnel
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel, Field

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["https://www.keplerv.com", "https://keplerv.com", "http://localhost:4399", "http://localhost:3000"], allow_methods=["GET", "POST"], allow_headers=["Content-Type"])

class ReconstructRequest(BaseModel):
    image_urls: list[str] = Field(min_length=1, max_length=6)
    steps: int = Field(default=64, ge=16, le=128)

@app.get("/health")
def health():
    return {"status": "ok", "model": "openai/shap-e-img2img", "max_images": 6}

@app.post("/reconstruct")
def reconstruct(req: ReconstructRequest):
    try:
        glb_bytes = reconstruct_to_glb(req.image_urls, steps=req.steps)
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e)) from e
    return Response(content=glb_bytes, media_type="model/gltf-binary")

nest_asyncio.apply()
public_url = ngrok.connect(8000, "http")
print("\n✅ Public API URL (paste into keplerv.com):", public_url)
print("   Health check:", str(public_url) + "/health\n")
uvicorn.run(app, host="0.0.0.0", port=8000)

### Troubleshooting
- **`assert torch.cuda.is_available()` fails** — enable the T4 GPU runtime.
- **ngrok authentication fails** — check the token in cell 4.
- **Cell 5 keeps running** — expected; it hosts the API while this notebook is open.
- **The result looks blobby or misses details** — Shap-E is a fast, low-detail diffusion model. The six exterior photos are arranged as one reference board, but the model does not truly fuse camera geometry like a photogrammetry system. Salvage damage and clutter make reconstruction harder.